# PyTorch for LLMs 01: Tensors and Autograd

> **What you finished last time:** one complete \`data -> module -> loss -> backward -> optimizer -> inference\` route.
> **The gap this chapter closes:** the loop worked, but axis meaning, dtype, device, and graph state were mostly implicit.
> **What this notebook delivers:** shape-first tensor evidence and a gradient audit for an LM-shaped computation.
> **What this chapter is not:** a survey of every tensor API.

**Guiding question:** Can you explain every tensor operation as selecting, regrouping, repeating, or contracting named axes?

| Complaint | Refinement |
|---|---|
| a printed tensor hides its roles | name every axis |
| indexing silently removes axes | compare integer index and slice |
| reshape can preserve values while breaking meaning | trace shape and value order |
| broadcasting feels magical | align trailing axes explicitly |
| autograd looks like a black box | compare graph result with a manual derivative |

\`\`\`mermaid
flowchart LR
    A["token IDs\n(batch, time)"] --> B["embedding lookup"]
    B --> C["states\n(batch, time, feature)"]
    C --> D["matrix multiply"]
    D --> E["logits\n(batch, time, vocab)"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Imports, reproducibility, and device -------------------------------------
import torch

torch.manual_seed(11)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={device}")


## 1. Shape, dtype, and device form one contract

Token IDs are integer addresses. Model states are floating-point measurements. Sending floating-point "IDs" into an embedding is not a harmless formatting choice.


In [ ]:
# -- Inspect scalar, vector, table, and sequence contracts --------------------
scalar = torch.tensor(5.5)
vector = torch.tensor([1.0, 2.0, 3.0])
token_ids = torch.tensor([[4, 1, 9, 2], [3, 8, 0, 7]], dtype=torch.long, device=device)
states = torch.randn(2, 4, 6, device=device)
for name, tensor in (("scalar", scalar), ("vector", vector), ("token_ids", token_ids), ("states", states)):
    print(f"{name:9s}: rank={tensor.ndim}, shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")


### Attempt 1: "take the first example"

**Predict:** Do \`states[0]\` and \`states[0:1]\` contain the same values? Yes. Do they have the same contract? No.


In [ ]:
# -- Measure integer indexing versus slicing ----------------------------------
one_example_without_batch = states[0]
one_example_batch = states[0:1]
same_values = torch.equal(one_example_without_batch, one_example_batch.squeeze(0))
print(f"states[0]   -> {tuple(one_example_without_batch.shape)} = (time, feature)")
print(f"states[0:1] -> {tuple(one_example_batch.shape)} = (batch, time, feature)")
print(f"Values match after explicit squeeze: {same_values}")


## 2. Reshape and transpose solve different problems

Reshape groups the same ordered values under a new address system. Transpose reorders axes.

\`\`\`text
states                     split heads                  move heads
(batch, time, feature) -> (batch, time, head, width) -> (batch, head, time, width)
(2,     4,    6)       -> (2,     4,    2,    3)     -> (2,     2,    4,    3)
\`\`\`

The value count stays 48. The axis story changes twice.


In [ ]:
# -- Prove value preservation while changing the address system ---------------
split_heads = states.reshape(2, 4, 2, 3)
heads_first = split_heads.transpose(1, 2)
restored = heads_first.transpose(1, 2).reshape(2, 4, 6)
print(f"states      {tuple(states.shape)}")
print(f"split_heads {tuple(split_heads.shape)}")
print(f"heads_first {tuple(heads_first.shape)}")
print(f"Round-trip values preserved: {torch.equal(states, restored)}")


## 3. Broadcasting is aligned repetition

**Predict:** Can a \`(feature,)\` bias be added to \`(batch, time, feature)\` states? Yes: trailing feature axes match. Can a \`(time,)\` vector be added directly? No: \`time=4\` collides with \`feature=6\`.

\`\`\`mermaid
flowchart LR
    A["states\n(2,4,6)"] --> C["add"]
    B["feature bias\n(6,)"] --> C
    C --> D["shifted states\n(2,4,6)"]
    E["time bias\n(4,)"] --> F["failure"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Show one valid broadcast and one visible shape failure -------------------
feature_bias = torch.arange(6, device=device, dtype=states.dtype)
shifted = states + feature_bias
time_bias = torch.arange(4, device=device, dtype=states.dtype)
try:
    states + time_bias
except RuntimeError as error:
    failure = str(error).splitlines()[0]
else:
    raise AssertionError("Expected incompatible trailing dimensions to fail")
fixed = states + time_bias.view(1, 4, 1)
print(f"Feature broadcast: {tuple(states.shape)} + {tuple(feature_bias.shape)} -> {tuple(shifted.shape)}")
print(f"Visible failure: {failure}")
print(f"Named repair: {tuple(states.shape)} + {tuple(time_bias.view(1, 4, 1).shape)} -> {tuple(fixed.shape)}")


## 4. Element-wise multiplication and matrix multiplication answer different questions

\`\`\`text
states @ W_vocab
(2,4,6) @ (6,10) -> (2,4,10)

The feature axis contracts. Batch and time survive. Vocabulary is appended.
\`\`\`


In [ ]:
# -- Compare scaling each coordinate with contracting the feature axis --------
gate = torch.linspace(0.5, 1.0, 6, device=device)
gated = states * gate
W_vocab = torch.randn(6, 10, device=device)
logits = states @ W_vocab
print(f"Element-wise: {tuple(states.shape)} * {tuple(gate.shape)} -> {tuple(gated.shape)}")
print(f"Matrix multiply: {tuple(states.shape)} @ {tuple(W_vocab.shape)} -> {tuple(logits.shape)}")


## 5. Autograd records influence, not intent

The graph knows which operations produced a scalar. It does not know whether your axis names or objective were sensible.

**Predict:** For \`y = (wx + b)^2\`, should \`w.grad\` equal \`2(wx+b)x\`? The next cell compares the recorded graph with that manual derivative.


In [ ]:
# -- Audit one autograd result against the chain rule -------------------------
x_value = torch.tensor(3.0)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
intermediate = w * x_value + b
objective = intermediate**2
objective.backward()
manual_dw = 2 * intermediate.detach() * x_value
manual_db = 2 * intermediate.detach()
print(f"autograd dw={w.grad.item():.1f}; manual dw={manual_dw.item():.1f}")
print(f"autograd db={b.grad.item():.1f}; manual db={manual_db.item():.1f}")
assert torch.allclose(w.grad, manual_dw) and torch.allclose(b.grad, manual_db)


### That worked, but gradients accumulate

Repeated \`backward()\` calls add responsibility into \`.grad\`. This is useful for intentional accumulation and dangerous when forgotten.


In [ ]:
# -- Measure accumulation, clearing, detach, and inference mode ---------------
weight = torch.tensor(2.0, requires_grad=True)
(weight * 3).backward()
first_gradient = weight.grad.item()
(weight * 3).backward()
accumulated_gradient = weight.grad.item()
weight.grad.zero_()
detached = (weight * 3).detach()
with torch.inference_mode():
    inferred = weight * 3
print(f"After one backward: {first_gradient:.1f}; after two: {accumulated_gradient:.1f}")
print(f"After zero_: {weight.grad.item():.1f}")
print(f"detach requires_grad={detached.requires_grad}; inference requires_grad={inferred.requires_grad}")


## 6. Language-model-shaped evidence

\`\`\`mermaid
flowchart LR
    A["IDs\nlong (B,T)"] --> B["Embedding table\n(V,D)"]
    B --> C["States\nfloat (B,T,D)"]
    C --> D["Vocabulary matrix\n(D,V)"]
    D --> E["Logits\n(B,T,V)"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Build an embedding-to-logits graph and inspect gradient shapes -----------
vocabulary_size = 10
embedding_size = 6
embedding = torch.nn.Embedding(vocabulary_size, embedding_size).to(device)
lm_head = torch.nn.Linear(embedding_size, vocabulary_size, bias=False).to(device)
embedded = embedding(token_ids)
lm_logits = lm_head(embedded)
toy_loss = lm_logits.square().mean()
toy_loss.backward()
print(f"IDs {tuple(token_ids.shape)} -> embeddings {tuple(embedded.shape)} -> logits {tuple(lm_logits.shape)}")
print(f"Embedding gradient shape: {tuple(embedding.weight.grad.shape)}")
print(f"LM-head gradient shape: {tuple(lm_head.weight.grad.shape)}")


### Your turn: change one known axis


In [ ]:
# -- CHANGE THIS: alter head count while preserving feature count -------------
YOUR_HEADS = 3
assert states.shape[-1] % YOUR_HEADS == 0
your_heads = states.reshape(2, 4, YOUR_HEADS, states.shape[-1] // YOUR_HEADS)
print(f"With {YOUR_HEADS} heads: {tuple(states.shape)} -> {tuple(your_heads.shape)}")


## Scorecard and coverage ledger

| Tier | Covered here |
|---|---|
| Built and measured | rank, shape, dtype, device, indexing, reshape, transpose, broadcasting, matmul, autograd, accumulation, detach, inference |
| Explained or illustrated | computation graphs as influence records; LM-shaped axis flow |
| Named and out of scope | sparse tensors, custom autograd functions, distributed devices, sequence window construction |

**Common failure modes:** unnamed axes, accidental batch removal, using reshape to hide a mismatch, floating-point token IDs, and stale accumulated gradients.

**Next:** Part 02 keeps these contracts visible while replacing raw tensor parameters with \`nn.Module\` and a complete training loop.
